# 00 – Setup: Create Virtual Environment & Prerequisites

This notebook creates an isolated Python virtual environment and walks through all prerequisites for this demo.

**Estimated time:** 5–10 minutes

---

## What you need

| Requirement | Notes |
|---|---|
| Python | `>= 3.10` |
| Azure CLI | `>= 2.55` |
| Bicep extension | `az bicep install` |
| Azure subscription | contributor access to create resources |

---

## Step 1 – Create Isolated Virtual Environment

Each demo has its own virtual environment to keep dependencies isolated and reproducible.

In [1]:
import subprocess
import sys
from pathlib import Path

# Determine venv location (in demo root)
demo_root = Path("..").resolve()
venv_path = demo_root / ".venv"
is_windows = sys.platform == "win32"
venv_python = venv_path / ("Scripts" if is_windows else "bin") / ("python.exe" if is_windows else "python")
venv_pip = venv_path / ("Scripts" if is_windows else "bin") / ("pip.exe" if is_windows else "pip")

print(f"Demo root: {demo_root}")
print(f"Virtual environment path: {venv_path}\n")

# Create venv if it doesn't exist
if not venv_path.exists():
    print("📦 Creating isolated virtual environment...")
    result = subprocess.run([sys.executable, "-m", "venv", str(venv_path)], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✓ Virtual environment created")
    else:
        print(f"ERROR creating venv: {result.stderr}")
        sys.exit(1)
else:
    print("✓ Virtual environment already exists")

# Verify venv python executable exists
if venv_python.exists():
    print(f"✓ Virtual environment ready")
else:
    print(f"ERROR: Python executable not found")
    sys.exit(1)

# Store venv paths for later cells
import os
os.environ['VENV_PYTHON'] = str(venv_python)
os.environ['VENV_PIP'] = str(venv_pip)

Demo root: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demo-template
Virtual environment path: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demo-template\.venv

📦 Creating isolated virtual environment...
✓ Virtual environment created
✓ Virtual environment ready


## Step 2 – Authenticate to Azure (If Needed)

In [ ]:
# Check existing login first to avoid unnecessary interactive prompt
account_check = subprocess.run(
    ['az', 'account', 'show', '--query', 'id', '-o', 'tsv'],
    capture_output=True,
    text=True
)

if account_check.returncode == 0 and account_check.stdout.strip():
    print("✓ Already authenticated with Azure CLI")
else:
    print("No active Azure login found. Starting device-code login flow...")
    print("Follow the instructions shown in the output.")
    # Stream output directly so users see progress and prompts
    login_result = subprocess.run(['az', 'login', '--use-device-code'])
    if login_result.returncode == 0:
        print("✓ Successfully logged in to Azure")
    else:
        print("ERROR: Azure login failed")
        sys.exit(1)

# Show available subscriptions
subprocess.run(['az', 'account', 'list', '--output', 'table'])

In [ ]:
# Optional: set a specific subscription if needed
# Leave SUBSCRIPTION_ID as None to keep current default subscription
SUBSCRIPTION_ID = None

if SUBSCRIPTION_ID:
    subprocess.run(['az', 'account', 'set', '--subscription', SUBSCRIPTION_ID], check=False)

subprocess.run(['az', 'account', 'show', '--output', 'table'])

## Step 3 – Install Python dependencies in virtual environment

Update `requirements.txt` in your demo folder to match the packages your demo needs.

In [ ]:
import os
from pathlib import Path

venv_pip = os.getenv('VENV_PIP')

if not venv_pip:
    print("ERROR: VENV_PIP is not set. Run Step 1 first.")
    sys.exit(1)

# Upgrade pip in venv
print("Upgrading pip in virtual environment...")
pip_upgrade = subprocess.run([venv_pip, 'install', '--upgrade', 'pip'])
if pip_upgrade.returncode == 0:
    print("✓ pip upgraded")
else:
    print("WARNING: pip upgrade returned a non-zero exit code")

# Install from requirements.txt with live output
print("\nInstalling dependencies from requirements.txt...")
requirements_file = Path("../requirements.txt")
if requirements_file.exists():
    install_result = subprocess.run([venv_pip, 'install', '-r', str(requirements_file)])
    if install_result.returncode == 0:
        print("✓ All packages installed successfully")
    else:
        print("ERROR: package installation failed")
        sys.exit(1)
else:
    print(f"ERROR: requirements.txt not found at {requirements_file}")
    sys.exit(1)

## Step 4 – Check .env file

`env/.env` is written automatically by `01_deploy_infra.ipynb` after deployment.
If you haven't deployed yet, this cell will tell you what to expect.

In [ ]:
import pathlib

env_path     = pathlib.Path("../env/.env")
example_path = pathlib.Path("../env/.env.example")

if env_path.exists():
    print("✅  env/.env found")
else:
    print("⚠️  env/.env not found — run 01_deploy_infra.ipynb first")
    print(f"\nExpected variables (from {example_path}):")
    print(example_path.read_text())

---

## ✓ Setup Complete

Virtual environment created and all dependencies installed in isolated environment.

**Next Steps:**
1. **Select the venv kernel in VS Code:**
   - Open VS Code command palette (Ctrl+Shift+P / Cmd+Shift+P)
   - Type "Python: Select Interpreter"
   - Choose `.venv` from the list
   
2. **Then run `01_deploy_infra.ipynb`** (or the next notebook for your demo)